In [10]:
import pandas as pd
import numpy as np
import string
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
df = pd.read_csv('emotions.txt', sep=';', names=['text', 'emotions'], header=0, on_bad_lines='skip')

print("First 10 rows:")
print(df.head(10))

print("\nShape of dataset:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

First 10 rows:
                                                text emotions
0          im updating my blog because i feel shitty  sadness
1  i never make her separate from me because i do...  sadness
2  i left with my bouquet of red and yellow tulip...      joy
3    i was feeling a little vain when i did this one  sadness
4  i cant walk into a shop anywhere where i do no...     fear
5   i felt anger when at the end of a telephone call    anger
6  i explain why i clung to a relationship with a...      joy
7  i like to have the same breathless feeling as ...      joy
8  i jest i feel grumpy tired and pre menstrual w...    anger
9                 i don t feel particularly agitated     fear

Shape of dataset: (1999, 2)

Missing values:
text        0
emotions    0
dtype: int64


In [4]:
print("Unique emotion labels:", df['emotions'].unique())

le = LabelEncoder()
df['emotion_encoded'] = le.fit_transform(df['emotions'])

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("\nEmotion to Number Mapping:")
print(mapping)

print("\nDataFrame with encoded labels:")
print(df.head())

Unique emotion labels: <ArrowStringArray>
['sadness', 'joy', 'fear', 'anger', 'love', 'surprise']
Length: 6, dtype: str

Emotion to Number Mapping:
{'anger': np.int64(0), 'fear': np.int64(1), 'joy': np.int64(2), 'love': np.int64(3), 'sadness': np.int64(4), 'surprise': np.int64(5)}

DataFrame with encoded labels:
                                                text emotions  emotion_encoded
0          im updating my blog because i feel shitty  sadness                4
1  i never make her separate from me because i do...  sadness                4
2  i left with my bouquet of red and yellow tulip...      joy                2
3    i was feeling a little vain when i did this one  sadness                4
4  i cant walk into a shop anywhere where i do no...     fear                1


In [5]:
df['text_lower'] = df['text'].str.lower()

print("5 samples BEFORE lowercasing:")
print(df['text'].head())

print("\n5 samples AFTER lowercasing:")
print(df['text_lower'].head())

print("""
Note: Lowercasing is important in NLP because it treats 'Happy' and 'happy' as the same word. 
This reduces vocabulary size and avoids duplicate features, improving model performance.
""")

5 samples BEFORE lowercasing:
0            im updating my blog because i feel shitty
1    i never make her separate from me because i do...
2    i left with my bouquet of red and yellow tulip...
3      i was feeling a little vain when i did this one
4    i cant walk into a shop anywhere where i do no...
Name: text, dtype: str

5 samples AFTER lowercasing:
0            im updating my blog because i feel shitty
1    i never make her separate from me because i do...
2    i left with my bouquet of red and yellow tulip...
3      i was feeling a little vain when i did this one
4    i cant walk into a shop anywhere where i do no...
Name: text_lower, dtype: str

Note: Lowercasing is important in NLP because it treats 'Happy' and 'happy' as the same word. 
This reduces vocabulary size and avoids duplicate features, improving model performance.



In [6]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['text_no_punct'] = df['text_lower'].apply(remove_punctuation)

print("5 examples BEFORE punctuation removal:")
print(df['text_lower'].head())

print("\n5 examples AFTER punctuation removal:")
print(df['text_no_punct'].head())

5 examples BEFORE punctuation removal:
0            im updating my blog because i feel shitty
1    i never make her separate from me because i do...
2    i left with my bouquet of red and yellow tulip...
3      i was feeling a little vain when i did this one
4    i cant walk into a shop anywhere where i do no...
Name: text_lower, dtype: str

5 examples AFTER punctuation removal:
0            im updating my blog because i feel shitty
1    i never make her separate from me because i do...
2    i left with my bouquet of red and yellow tulip...
3      i was feeling a little vain when i did this one
4    i cant walk into a shop anywhere where i do no...
Name: text_no_punct, dtype: str


In [7]:
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

df['text_no_num'] = df['text_no_punct'].apply(remove_numbers)

print("5 examples BEFORE number removal:")
print(df['text_no_punct'].head())

print("\n5 examples AFTER number removal:")
print(df['text_no_num'].head())

5 examples BEFORE number removal:
0            im updating my blog because i feel shitty
1    i never make her separate from me because i do...
2    i left with my bouquet of red and yellow tulip...
3      i was feeling a little vain when i did this one
4    i cant walk into a shop anywhere where i do no...
Name: text_no_punct, dtype: str

5 examples AFTER number removal:
0            im updating my blog because i feel shitty
1    i never make her separate from me because i do...
2    i left with my bouquet of red and yellow tulip...
3      i was feeling a little vain when i did this one
4    i cant walk into a shop anywhere where i do no...
Name: text_no_num, dtype: str


In [8]:
def remove_non_ascii(text):
    return text.encode('ascii', 'ignore').decode('ascii')

df['text_ascii'] = df['text_no_num'].apply(remove_non_ascii)

print("5 examples after removing emojis/special chars:")
print(df['text_ascii'].head())

5 examples after removing emojis/special chars:
0            im updating my blog because i feel shitty
1    i never make her separate from me because i do...
2    i left with my bouquet of red and yellow tulip...
3      i was feeling a little vain when i did this one
4    i cant walk into a shop anywhere where i do no...
Name: text_ascii, dtype: str


In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = word_tokenize(text)
    filtered = [word for word in words if word not in stop_words and word != '']
    return ' '.join(filtered)

df['text_no_stop'] = df['text_ascii'].apply(remove_stopwords)

print("5 cleaned text samples after stopword removal:")
print(df['text_no_stop'].head())

5 cleaned text samples after stopword removal:
0                         im updating blog feel shitty
1      never make separate ever want feel like ashamed
2    left bouquet red yellow tulips arm feeling sli...
3                              feeling little vain one
4           cant walk shop anywhere feel uncomfortable
Name: text_no_stop, dtype: str


In [ ]:
def clean_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # 3. Remove numbers
    text = re.sub(r'\d+', '', text)
    # 4. Remove emojis/special chars - keep ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')
    # 5. Remove stopwords
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words and word != '']
    return ' '.join(words)

df['cleaned_text'] = df['text'].apply(clean_text)

print("Original vs Cleaned - 10 samples:")
print(df[['text', 'cleaned_text']].head(10))

In [ ]:
df['word_count'] = df['cleaned_text'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10,5))
sns.histplot(df['word_count'], bins=30, kde=True)
plt.title('Distribution of Cleaned Text Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.show()

print("Average text length:", round(df['word_count'].mean(), 2))
print("Min text length:", df['word_count'].min())
print("Max text length:", df['word_count'].max())

print("""
Observations: 
1. Most texts are between 3-12 words after cleaning.
2. Very long texts are rare. 
3. Data is now ready for vectorization like TF-IDF or Word2Vec.
""")

In [ ]:
cleaned_df = df[['cleaned_text', 'emotions', 'emotion_encoded']]
cleaned_df.to_csv('cleaned_emotions.csv', index=False)
print("Saved as cleaned_emotions.csv")

print("\nValue counts of each emotion:")
print(df['emotions'].value_counts())